In [1]:
# Modules
import numpy as np
import pandas as pd
import itertools
import destruction_models as models
import tensorflow as tf
import random

from destruction_utilities import *
from destruction_statistics import *
from numpy import random
import matplotlib.pyplot as plt
from tensorflow.keras import callbacks, preprocessing
from tensorflow.keras.utils import Sequence
from tensorflow.keras.metrics import CategoricalAccuracy, Precision, AUC
from tensorflow.keras.models import load_model
from sklearn.metrics import precision_recall_curve, roc_auc_score
from os import path
import zarr
import shutil
from tensorflow.keras import backend as K
import gc
import time

In [2]:
CITY = 'aleppo'

In [3]:
def get_zarr(city, type, set, balanced = False):
    if balanced:
        path = f'../data/{city}/others/{city}_{type}s_{set}_balanced.zarr'
    else:
        path = f'../data/{city}/others/{city}_{type}s_{set}.zarr'
    return zarr.open(path, mode='r')


def reshuffle_data(X, y): 
    print("Shuffling data started...")
    shutil.rmtree('X_shuffled.zarr', ignore_errors=True)
    shutil.rmtree('y_shuffled.zarr', ignore_errors=True)
    n=X.shape[0]
    
    tuple_pair = make_tuple_pair(n, 250)
    
    np.random.shuffle(tuple_pair)
    zarr.save("X1.zarr", np.empty((0,128,128,3)))
    zarr.save("y1.zarr", np.empty((0,1,1,1)))
    
    X1 = zarr.open("X1.zarr", mode='a')
    y1 = zarr.open("y1.zarr", mode='a')
    
    print(f"Reordering array in batches of 250. Total {len(tuple_pair)} sets..")
    for i, t in enumerate(tuple_pair):
        if i % 50 == 0 and i!=0:
            print(f"Finished {i} sets")
        Xs = X[t[0]:t[1]]
        ys = y[t[0]:t[1]]
        X1.append(Xs)
        y1.append(ys)
    
    
    zarr.save("X_shuffled.zarr", np.empty((0,128,128,3)))
    zarr.save("y_shuffled.zarr", np.empty((0,1,1,1)))
    
    X_shuffled = zarr.open("X_shuffled.zarr", mode='a')
    y_shuffled = zarr.open("y_shuffled.zarr", mode='a')
    
    tuple_pair = make_tuple_pair(n, 10000)
    print(f"Shuffling array in batches of 10000. Total {len(tuple_pair)} sets..")
    for i, t in enumerate(tuple_pair):
        if i % 5 == 0 and i != 0:
            print(f"Finished {i} sets")
        shuffled = np.arange(0, 10000)
        np.random.shuffle(shuffled)
        Xs = X1[t[0]:t[1]][shuffled]
        ys = y1[t[0]:t[1]][shuffled]
        X_shuffled.append(Xs)
        y_shuffled.append(ys)
        
    shutil.rmtree('X1.zarr')
    shutil.rmtree('y1.zarr')
    return X_shuffled, y_shuffled

def recall_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + K.epsilon())
    return recall

def precision_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision

def f1_m(y_true, y_pred):
    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+K.epsilon()))

auc = AUC(
    num_thresholds=200,
    curve='ROC',
)
    
def run_model(training_images, training_labels, validation_images, validation_labels, epochs=50):
    training_generator = ZarrGenerator(training_images, training_labels, batch_size=32)
    validation_generator = ZarrGenerator(validation_images, validation_labels, batch_size=32)
    
    training_callbacks = [
    #     callbacks.EarlyStopping(monitor='val_auc', restore_best_weights=True, patience=5),
    #     callbacks.BackupAndRestore(backup_dir='../models')
    ]

    args  = dict(shape=(128, 128, 3), filters=16, units=32, dropout=0.1) # ! Check parameters before run
    model = models.convolutional_network(**args)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', f1_m, precision_m, recall_m, auc ])

    # Train model on dataset
    model.fit_generator(
        training_generator,
        validation_data=validation_generator, 
        epochs=epochs, 
        callbacks = training_callbacks
    )
    
    return model


def calculate_auc(model, test_images, test_labels):
    gc.collect(generation=2)    
    batch_size = 5000
    iters = test_images.shape[0] // batch_size
    preds = []
    labels = []
    for i in range(0, iters):
        end = (i+1)*batch_size
        if i == iters - 1:
            preds.append(model.predict(test_images[i*batch_size:]))
            labels.append(test_labels[i*batch_size:])
        else:
            preds.append(model.predict(test_images[i*batch_size: end]))
            labels.append(test_labels[i*batch_size: end])
            
    yhat = np.squeeze(np.concatenate(preds, axis=0))
    y = np.squeeze(np.concatenate(labels, axis=0 ))
    roc_auc = roc_auc_score(y, yhat)
    
    return roc_auc
    

Metal device set to: Apple M1


2022-06-19 15:58:07.737354: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2022-06-19 15:58:07.737527: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [4]:
class ZarrGenerator(Sequence):
    def __init__(self, images, labels, batch_size=32):
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        
    def __len__(self):
        return len(self.images)//self.batch_size
    
    def __getitem__(self, index):

        X = self.images[index*self.batch_size:(index+1)*self.batch_size]
        y = self.labels[index*self.batch_size:(index+1)*self.batch_size]
        
        return self.augment(X), y.flatten()
    
    def augment(self, X):
        # Horizontal and vertical flip
        flipping_funcs = [
            lambda image: image,
            lambda image: np.fliplr(image),
            lambda image: np.flipud(image),
            lambda image: np.flipud(np.fliplr(image))
        ]
        func = random.choice(flipping_funcs)
        X = func(X)
        
        # Brightness
        alpha = random.choice(np.linspace(0.85, 1.4))
        X = X * alpha
        
        return X

In [6]:
training_images = get_zarr(CITY, 'image', 'train', balanced=True)
training_labels = get_zarr(CITY, 'label', 'train', balanced=True)
validation_images = get_zarr(CITY, 'image', 'validate')
validation_labels = get_zarr(CITY, 'label', 'validate')
test_images = get_zarr(CITY, 'image', 'test')
test_labels = get_zarr(CITY, 'label', 'test')

In [7]:
print(training_images.shape, training_labels.shape, np.unique(training_labels, return_counts=True))
print(validation_images.shape, validation_labels.shape, np.unique(validation_labels, return_counts=True))
print(test_images.shape, test_labels.shape, np.unique(test_labels, return_counts=True))

(140000, 128, 128, 3) (140000, 1, 1, 1) (array([0., 1.]), array([69931, 70069]))
(15804, 128, 128, 3) (15804, 1, 1, 1) (array([0., 1.]), array([15064,   740]))
(15712, 128, 128, 3) (15712, 1, 1, 1) (array([0., 1.]), array([15144,   568]))


In [8]:
for i in range(0,50):
    training_images_shuffled, training_labels_shuffled = reshuffle_data(training_images, training_labels)
    print("Shuffling complete..")
    model = run_model(training_images_shuffled, training_labels_shuffled, validation_images, validation_labels, epochs=50)
    print("Model optimization complete..")
    auc = calculate_auc(model, test_images, test_labels)
    
    if(auc > 0.8):
        ts = str(time.time())
        model.save(f'../models/{CITY}_{ts}', save_format="h5")

Shuffling data started...
Reordering array in batches of 250. Total 560 sets..
Finished 50 sets
Finished 100 sets
Finished 150 sets
Finished 200 sets
Finished 250 sets
Finished 300 sets
Finished 350 sets
Finished 400 sets
Finished 450 sets
Finished 500 sets
Finished 550 sets
Shuffling array in batches of 10000. Total 14 sets..
Finished 5 sets
Finished 10 sets


OSError: [Errno 28] No space left on device